# German Industry Run-Ups, Crashes, and Risk-Free-Adjusted Returns

This notebook is the executable Python version of the revised Stata workflow. It performs the Germany/common-equity sample construction, EUR/DEM normalization, daily cleaning, monthly security returns, value-weighted industry portfolios, run-up identification, forward crash outcomes, volatility controls, and risk-free-adjusted summary table.

The notebook uses `comp_global_daily.dta` as the raw source and `germany_rf.csv` as the monthly German three-month interest-rate series. Existing Stata outputs are treated as supporting files, not as substitutes for the calculations below.

## 1. Setup and file discovery

All paths are discovered relative to the notebook/project folder. The code does not rely on a hard-coded Windows drive. The raw file can be large, so the loader reads only required variables and processes the Stata file in chunks.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, Image

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'germany_rf.csv').exists():
    for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
        if (candidate / 'germany_rf.csv').exists():
            PROJECT_ROOT = candidate
            break

RAW_FILE = PROJECT_ROOT / 'comp_global_daily.dta'
RF_FILE = PROJECT_ROOT / 'germany_rf.csv'
OUTPUT_DIR = PROJECT_ROOT / 'notebook_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
print('Project:', PROJECT_ROOT)
print('Raw file:', RAW_FILE, RAW_FILE.exists())
print('Risk-free file:', RF_FILE, RF_FILE.exists())
if not RF_FILE.exists():
    raise FileNotFoundError('germany_rf.csv was not found in the project folder or its parents.')


The previous `19900001` error occurred because an integer monthly index was passed to a `YYYYMMDD` parser. The helper below deliberately separates daily dates from monthly periods. Invalid dates become missing and are reported instead of stopping the notebook.

In [ ]:
def parse_daily_date(series):
    s = series.astype('string').str.strip()
    out = pd.to_datetime(s, errors='coerce')
    bad = int(out.isna().sum())
    if bad:
        print(f'Invalid daily dates converted to NaT: {bad}')
    return out

def month_period_from_year_month(year, month):
    y = pd.to_numeric(year, errors='coerce')
    m = pd.to_numeric(month, errors='coerce')
    valid = y.between(1900, 2100) & m.between(1, 12)
    result = pd.Series(pd.NaT, index=y.index, dtype='period[M]')
    result.loc[valid] = pd.PeriodIndex(
        year=y.loc[valid].astype(int), month=m.loc[valid].astype(int), freq='M'
    )
    return result

def safe_month_period(series):
    if pd.api.types.is_period_dtype(series):
        return series.astype('period[M]')
    parsed = pd.to_datetime(series, errors='coerce')
    return parsed.dt.to_period('M')


## 2. Load and clean the Germany security sample

The raw Stata file is read in chunks. We immediately retain Germany, common equity, valid industry classifications, and positive prices. This avoids loading unnecessary countries and variables into memory.

In [ ]:
required_cols = [
    'fic','gvkey','iid','datadate','conm','gsector','ajexdi','cshoc',
    'curcdd','prccd','trfd','tpci'
]
available_cols = None
if RAW_FILE.exists():
    probe = pd.read_stata(RAW_FILE, iterator=True, chunksize=1)
    available_cols = probe._native_reader.varlist
    probe.close()
    missing = sorted(set(required_cols) - set(available_cols))
    if missing:
        raise ValueError(f'Missing required raw columns: {missing}')

def load_germany_security_data(path, chunksize=250_000):
    pieces = []
    reader = pd.read_stata(path, columns=required_cols, iterator=True, chunksize=chunksize)
    for i, chunk in enumerate(reader, start=1):
        chunk = chunk.loc[(chunk['fic'] == 'DEU') & (chunk['tpci'].astype('string') == '0')].copy()
        chunk = chunk.loc[chunk['gsector'].notna() & (pd.to_numeric(chunk['prccd'], errors='coerce') > 0)]
        if not chunk.empty:
            pieces.append(chunk)
        if i % 10 == 0:
            print(f'Processed raw chunks: {i}')
    if not pieces:
        raise ValueError('No Germany observations survived the sample filters.')
    out = pd.concat(pieces, ignore_index=True)
    out['datadate'] = parse_daily_date(out['datadate'])
    out = out.loc[out['datadate'].notna()].copy()
    for col in ['prccd','ajexdi','cshoc','trfd']:
        out[col] = pd.to_numeric(out[col], errors='coerce')
    out['fx_eur'] = np.where(out['curcdd'].eq('EUR'), 1.0, np.where(out['curcdd'].eq('DEM'), 1/1.95583, np.nan))
    out = out.loc[out['fx_eur'].notna()].copy()
    return out.sort_values(['gvkey','iid','datadate']).reset_index(drop=True)

if RAW_FILE.exists():
    securities = load_germany_security_data(RAW_FILE)
    display(securities.head())
    print('Filtered security-date rows:', len(securities))
else:
    raise FileNotFoundError('comp_global_daily.dta is required to execute the full notebook pipeline.')


## 3. Total-return prices and daily return cleaning

Prices are converted into EUR before returns are calculated. The security key is `gvkey + iid`; grouping by both identifiers prevents a return from one security being used as the lag for another security with the same company identifier.

In [ ]:
g = securities.groupby(['gvkey','iid'], sort=False)
securities['tr_price_eur'] = (securities['prccd'] * securities['fx_eur'] / securities['ajexdi']) * securities['trfd']
securities['tr_price_lag'] = g['tr_price_eur'].shift(1)
securities['ret_daily'] = securities['tr_price_eur'].div(securities['tr_price_lag']).sub(1).where(securities['tr_price_lag'] > 0)
securities['tr_price_lag2'] = g['tr_price_eur'].shift(2)
securities['ret_2day'] = securities['tr_price_eur'].div(securities['tr_price_lag2']).sub(1).where(securities['tr_price_lag2'] > 0)
prev_ret = g['ret_daily'].shift(1)
prev_gvkey = g['gvkey'].shift(1)
prev_iid = g['iid'].shift(1)
securities['bad_reversal'] = ((securities['ret_daily'] > 1) | (prev_ret > 1)) & (securities['ret_2day'] < .20)
securities['bad_extreme'] = securities['ret_daily'] > 2
securities['bad_return'] = securities['bad_reversal'] | securities['bad_extreme']
bad_count = int(securities['bad_return'].sum())
securities.loc[securities['bad_return'], 'ret_daily'] = np.nan
print('Flagged daily return observations:', bad_count)
securities.to_pickle(OUTPUT_DIR / 'germany_daily_clean.pkl')


The cleaning step does not winsorize returns. It sets only observations that meet the reversal/extreme-return rules to missing. This preserves plausible large movements while preventing obvious data errors from contaminating monthly portfolios and volatility.

## 4. Monthly security returns and value-weighted industry portfolios

The last available trading day in each security-month is retained. Monthly returns require an immediately preceding calendar month. Industry returns use lagged market capitalization as the portfolio weight, and the German market return uses the same underlying securities.

In [ ]:
securities['mdate'] = securities['datadate'].dt.to_period('M')
monthly = (securities.sort_values(['gvkey','iid','mdate','datadate'])
           .groupby(['gvkey','iid','mdate'], as_index=False, sort=False).tail(1).copy())
monthly = monthly.sort_values(['gvkey','iid','mdate'])
monthly['tr_price_month_lag'] = monthly.groupby(['gvkey','iid'])['tr_price_eur'].shift(1)
monthly['mdate_lag'] = monthly.groupby(['gvkey','iid'])['mdate'].shift(1)
monthly['ret_monthly'] = (monthly['tr_price_eur'] / monthly['tr_price_month_lag'] - 1).where(
    (monthly['tr_price_month_lag'] > 0) & (monthly['mdate'] == monthly['mdate_lag'] + 1)
)
monthly['mktcap_eur'] = monthly['prccd'] * monthly['fx_eur'] * monthly['cshoc']
monthly['mktcap_lag'] = monthly.groupby(['gvkey','iid'])['mktcap_eur'].shift(1)
monthly = monthly.loc[monthly['gsector'].notna() & monthly['ret_monthly'].notna() & (monthly['mktcap_lag'] > 0)].copy()
monthly['industry_mktcap'] = monthly.groupby(['mdate','gsector'])['mktcap_lag'].transform('sum')
monthly['industry_weight'] = monthly['mktcap_lag'] / monthly['industry_mktcap']
monthly['industry_contribution'] = monthly['industry_weight'] * monthly['ret_monthly']
monthly['industry_return'] = monthly.groupby(['mdate','gsector'])['industry_contribution'].transform('sum')
monthly['market_mktcap'] = monthly.groupby('mdate')['mktcap_lag'].transform('sum')
monthly['market_return'] = monthly.groupby('mdate').apply(lambda x: np.nan, include_groups=False) if False else monthly['ret_monthly']
market = monthly.groupby('mdate').apply(lambda x: np.average(x['ret_monthly'], weights=x['mktcap_lag'])).rename('market_return').reset_index()
industry = monthly[['mdate','gsector','industry_return']].drop_duplicates().merge(market, on='mdate', how='left')
industry['net_market_return'] = industry['industry_return'] - industry['market_return']
industry = industry.sort_values(['gsector','mdate']).reset_index(drop=True)
industry.to_pickle(OUTPUT_DIR / 'germany_industry_monthly.pkl')
display(industry.head())
print('Industry-month rows:', len(industry))


## 5. Load and merge the German risk-free series

`IR3TIB01DEM156N` is a monthly German three-month interbank rate reported in percent per year. The notebook converts percent to decimal and divides by 12 to obtain a simple monthly rate. This convention is stated explicitly so the risk-free adjustment is reproducible.

In [ ]:
rf = pd.read_csv(RF_FILE)
required_rf = {'observation_date','IR3TIB01DEM156N'}
missing_rf = required_rf - set(rf.columns)
if missing_rf:
    raise ValueError(f'Risk-free CSV is missing columns: {sorted(missing_rf)}')
rf['observation_date'] = pd.to_datetime(rf['observation_date'], errors='coerce')
if rf['observation_date'].isna().all():
    raise ValueError('The risk-free CSV has no valid observation dates.')
rf['rf_annual_decimal'] = pd.to_numeric(rf['IR3TIB01DEM156N'], errors='coerce') / 100
rf['rf_monthly'] = rf['rf_annual_decimal'] / 12
rf['mdate'] = rf['observation_date'].dt.to_period('M')
rf = rf.dropna(subset=['mdate']).sort_values('mdate').drop_duplicates('mdate', keep='last')
print('RF observations:', len(rf))
print('RF date range:', rf['observation_date'].min().date(), 'to', rf['observation_date'].max().date())

industry = industry.merge(rf[['mdate','rf_monthly']], on='mdate', how='left', validate='many_to_one')
print('Industry-month observations missing RF:', int(industry['rf_monthly'].isna().sum()))
print(industry[industry['mdate'].isin([pd.Period('1998-12','M'), pd.Period('1999-01','M')])][['mdate','rf_monthly']].drop_duplicates())
industry['industry_net_rf'] = industry['industry_return'] - industry['rf_monthly']
industry['market_net_rf'] = industry['market_return'] - industry['rf_monthly']


The merge is performed by monthly `Period` values, not by converting a Stata month index into a fake day. The December 1998/January 1999 diagnostic is included because it is the main historical currency-transition area in the German sample.

## 6. Run-up indicators, forward outcomes, and maximum drawdown

The strict run-up definition requires prior 24-month raw and net-market returns above the threshold and prior 60-month raw returns of at least 50%. The signal is lagged one month. Forward outcomes require consecutive monthly observations through month +24.

In [ ]:
def add_window_features(x):
    x = x.sort_values('mdate').copy()
    periods = x['mdate'].astype('period[M]')
    complete = periods.diff().eq(1)
    x['raw_24'] = (1 + x['industry_return']).rolling(24, min_periods=24).apply(np.prod, raw=True) - 1
    x['net_24'] = (1 + x['net_market_return']).rolling(24, min_periods=24).apply(np.prod, raw=True) - 1
    x['raw_60'] = (1 + x['industry_return']).rolling(60, min_periods=60).apply(np.prod, raw=True) - 1
    x['netrf_12'] = (1 + x['industry_net_rf']).rolling(12, min_periods=12).apply(np.prod, raw=True) - 1
    x['netrf_24'] = (1 + x['industry_net_rf']).rolling(24, min_periods=24).apply(np.prod, raw=True) - 1
    x['market_net_rf_12'] = (1 + x['market_net_rf']).rolling(12, min_periods=12).apply(np.prod, raw=True) - 1
    x['market_net_rf_24'] = (1 + x['market_net_rf']).rolling(24, min_periods=24).apply(np.prod, raw=True) - 1
    x['runup_raw24'] = x['raw_24'].shift(1)
    x['runup_net24'] = x['net_24'].shift(1)
    x['runup_raw60'] = x['raw_60'].shift(1)
    x['complete24'] = True
    for j in range(1,25):
        x['complete24'] &= periods.shift(-j).eq(periods + j)
    x['fwd12_raw'] = (1 + x['industry_return']).shift(-1).rolling(12, min_periods=12).apply(np.prod, raw=True).shift(-11) - 1
    x['fwd24_raw'] = (1 + x['industry_return']).shift(-1).rolling(24, min_periods=24).apply(np.prod, raw=True).shift(-23) - 1
    return x

industry = industry.groupby('gsector', group_keys=False).apply(add_window_features).reset_index(drop=True)
for threshold in [50,75,100,125,150]:
    q = (industry['runup_raw24'].ge(threshold/100) & industry['runup_net24'].ge(threshold/100) & industry['runup_raw60'].ge(.50))
    prior = q.groupby(industry['gsector']).transform(lambda s: s.shift(1).rolling(24, min_periods=1).max().fillna(0).astype(bool))
    industry[f'runup{threshold}'] = q & ~prior

def maximum_drawdown(path):
    if len(path) == 0: return np.nan
    wealth = np.cumprod(1 + np.asarray(path, dtype=float))
    wealth = np.r_[1.0, wealth]
    return np.min(wealth / np.maximum.accumulate(wealth) - 1)

def add_forward_drawdowns(x):
    x = x.sort_values('mdate').copy()
    x['maxdd_raw24'] = np.nan
    x['maxdd_net24'] = np.nan
    for i in range(len(x)):
        if i + 24 < len(x) and x.iloc[i+1:i+25]['mdate'].diff().dropna().eq(1).all():
            x.iloc[i, x.columns.get_loc('maxdd_raw24')] = maximum_drawdown(x.iloc[i+1:i+25]['industry_return'])
            x.iloc[i, x.columns.get_loc('maxdd_net24')] = maximum_drawdown(x.iloc[i+1:i+25]['net_market_return'])
    x['crash_raw40'] = x['maxdd_raw24'].le(-.40)
    x['crash_net40'] = x['maxdd_net24'].le(-.40)
    return x

industry = industry.groupby('gsector', group_keys=False).apply(add_forward_drawdowns).reset_index(drop=True)
print('Strict 100% episodes:', int(industry['runup100'].sum()))
print('Complete 24-month outcomes:', int(industry['complete24'].sum()))


The drawdown calculation includes event month 0 as the initial wealth level of one. A crash is therefore a 40% peak-to-trough loss during the following 24 months, even if the endpoint return later recovers.

## 7. Risk-free-adjusted run-up and crash table

This table is the Germany-focused version of the requested price-run-up/crash summary. It uses the German monthly three-month rate, reports the number of distinct episodes, and separates return statistics, subsequent crashes, and subsequent booms. A boom is defined here as a complete 24-month forward net-of-risk-free industry return of at least 50%.

In [ ]:
def table_row(label, subset, episode_count=None):
    subset = subset.loc[subset['complete24'] & subset['netrf_24'].notna()].copy()
    if episode_count is None: episode_count = len(subset)
    boom = subset['fwd24_raw'].notna() & ((1 + subset['industry_net_rf']).rolling(24, min_periods=24).apply(np.prod, raw=True).shift(-23) - 1 >= .50)
    crash = subset['crash_raw40'].fillna(False)
    crash_drawdown = subset.loc[crash, 'maxdd_raw24'].mean() * 100 if crash.any() else np.nan
    return {
        'Pick-up threshold (%)': label,
        'Number of run-ups identified': episode_count,
        '12-month net risk-free return (%)': subset['netrf_12'].mean() * 100,
        '24-month net risk-free return (%)': subset['netrf_24'].mean() * 100,
        '12-month net-market return (%)': subset['market_net_rf_12'].mean() * 100,
        '24-month net-market return (%)': subset['market_net_rf_24'].mean() * 100,
        'Std. dev. 24-month net risk-free return (%)': subset['netrf_24'].std() * 100,
        'Skewness 24-month net risk-free return': subset['netrf_24'].skew(),
        'Kurtosis 24-month net risk-free return': subset['netrf_24'].kurt(),
        'Crashes (%)': crash.mean() * 100,
        'Drawdown of crashes (%)': crash_drawdown,
        'Booms (%)': boom.mean() * 100,
        '24-month net risk-free return of booms (%)': subset.loc[boom, 'netrf_24'].mean() * 100 if boom.any() else np.nan
    }

rows = []
for threshold in [50,75,100,125,150]:
    events = industry.loc[industry[f'runup{threshold}']].copy()
    rows.append(table_row(threshold, events, len(events)))
rf_table = pd.DataFrame(rows)
rf_table = rf_table.round(3)
display(Markdown('### German industry run-ups and subsequent outcomes'))
display(rf_table)
rf_table.to_excel(OUTPUT_DIR / 'germany_price_runups_crashes_rf.xlsx', index=False)
print('Saved:', OUTPUT_DIR / 'germany_price_runups_crashes_rf.xlsx')


The table is intentionally structured with return statistics, subsequent crashes, and subsequent booms. The German sample is much smaller than the reference table, so the number of identified episodes must always be displayed alongside the percentages. Missing values mean that the relevant complete forward window or subgroup does not exist, not that the return was zero.

## 8. Regression and event-study outputs

The existing Stata-generated regression and event-time files remain available for comparison. They should be read only after the calculations above have completed. The risk-free-adjusted table generated here is a new notebook output and does not overwrite the Stata files.

In [ ]:
supporting = [
    'germany_crash_results.xlsx', 'germany_crash_test.xlsx',
    'germany_footnote15_regressions.xlsx', 'germany_event_time_paths.xlsx',
    'germany_table1_paper_style.xlsx', 'germany_table3_main.xlsx'
]
for filename in supporting:
    path = PROJECT_ROOT / filename
    print(filename, 'FOUND' if path.exists() else 'MISSING')
    if path.exists() and filename.endswith('.xlsx'):
        display(pd.read_excel(path).head())


## 9. Final validation

This final cell checks the required source files, date validity, risk-free merge coverage, required columns, and saved output. Run the notebook from a fresh kernel using **Restart Kernel and Run All**.

In [ ]:
required_output_columns = {
    'Pick-up threshold (%)', 'Number of run-ups identified',
    '12-month net risk-free return (%)', '24-month net risk-free return (%)',
    '12-month net-market return (%)', '24-month net-market return (%)',
    'Crashes (%)', 'Drawdown of crashes (%)', 'Booms (%)'
}
assert RF_FILE.exists()
assert industry['mdate'].notna().all(), 'Malformed monthly periods remain.'
assert rf_table.columns.isin(required_output_columns).sum() == len(required_output_columns)
assert (OUTPUT_DIR / 'germany_price_runups_crashes_rf.xlsx').exists()
print('Validation passed.')
print('Risk-free merge missing observations:', int(industry['rf_monthly'].isna().sum()))
print('Risk-free-adjusted output:', OUTPUT_DIR / 'germany_price_runups_crashes_rf.xlsx')
